# College Basketball Moneyline Model

A complete moneyline model for men's college basketball, tested the honest way.

1. **Data**: every game since 2012 with box scores, and moneylines since 2013
2. **Cleaning**: price checks, including a trap that fakes profits if you miss it
3. **The market by itself**: how favorites and underdogs have actually paid
4. **Ratings**: opponent-adjusted efficiency, tempo and Elo for every team
5. **The model**: its own win probability, then blended with the moneyline and the point spread
6. **The test**: rules fixed on 2017–2021, then judged once on 2022 onward
7. **Upcoming games**: probabilities and edges for every game with a line
8. **Saved files** for the website

**How to use it**

1. Put `cbb_data.zip` in the Files panel (folder icon on the left, then the upload button). It is the data file from `cbb_data_pull.ipynb`.
2. Switch on your `CBBD_API_KEY` secret for this notebook (key icon on the left, then "Notebook access"). The last three weeks are then refreshed automatically so the picks are current, for about 10 API calls.
3. Click **Runtime → Run all**. It takes about two minutes.

Either step 1 or step 2 is enough. With only the key, the notebook downloads everything itself, which takes about 12 minutes.

In [ ]:
import os, io, json, time, zipfile, warnings
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


# ---- Odds math ----
def american_to_decimal(odds):
    odds = np.asarray(odds, dtype=float)
    size = np.abs(odds)
    return np.where(odds < 0, 1 + 100 / size, 1 + size / 100)


def implied_prob(odds):
    odds = np.asarray(odds, dtype=float)
    size = np.abs(odds)
    return np.where(odds < 0, size / (size + 100), 100 / (size + 100))


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def logloss(p, y):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, float)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def brier(p, y):
    return float(((np.asarray(p, float) - np.asarray(y, float)) ** 2).mean())

---
# 1. Data

Three tables from CollegeBasketballData.com:

- **games**: date, site, teams, final score
- **team_stats**: each team's box score in each game (possessions, three-point attempts and makes)
- **lines**: spread, total and moneylines, one row per game per provider

Seasons are named by the year they end in, so 2023–24 is `2024`.

In [ ]:
DATA_ZIP = "cbb_data.zip"
TABLES = ("cbb_games.parquet", "cbb_team_stats.parquet", "cbb_lines.parquet")
API = "https://api.collegebasketballdata.com"


def get_key():
    key = os.environ.get("CBBD_API_KEY")
    if key:
        return key.strip()
    try:
        from google.colab import userdata
        return userdata.get("CBBD_API_KEY").strip()
    except Exception:
        return None


def nested(d, *keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d


def flatten_games(raw):
    return pd.DataFrame([{
        "game_id": r.get("id"), "source_id": r.get("sourceId"), "season": r.get("season"), "season_type": r.get("seasonType"),
        "start_date": r.get("startDate"), "status": r.get("status"), "neutral_site": r.get("neutralSite"),
        "conference_game": r.get("conferenceGame"), "tournament": r.get("tournament"), "game_type": r.get("gameType"),
        "home_team_id": r.get("homeTeamId"), "home_team": r.get("homeTeam"), "home_conference": r.get("homeConference"),
        "home_points": r.get("homePoints"),
        "away_team_id": r.get("awayTeamId"), "away_team": r.get("awayTeam"), "away_conference": r.get("awayConference"),
        "away_points": r.get("awayPoints")} for r in raw])


def flatten_stats(raw):
    def unit(r, side, prefix):
        s = r.get(side) or {}
        return {f"{prefix}points": nested(s, "points", "total"), f"{prefix}possessions": s.get("possessions"),
                f"{prefix}fg3a": nested(s, "threePointFieldGoals", "attempted"), f"{prefix}fg3m": nested(s, "threePointFieldGoals", "made")}
    return pd.DataFrame([{"game_id": r.get("gameId"), "season": r.get("season"), "start_date": r.get("startDate"),
                          "team_id": r.get("teamId"), "team": r.get("team"), **unit(r, "teamStats", "")} for r in raw])


def flatten_lines(raw):
    return pd.DataFrame([{
        "game_id": r.get("gameId"), "season": r.get("season"), "start_date": r.get("startDate"),
        "provider": l.get("provider"), "spread": l.get("spread"), "over_under": l.get("overUnder"),
        "home_moneyline": l.get("homeMoneyline"), "away_moneyline": l.get("awayMoneyline"),
        "spread_open": l.get("spreadOpen"), "over_under_open": l.get("overUnderOpen")}
        for r in raw for l in (r.get("lines") or [])])


def pull(path, start, end, headers):
    # All rows between two dates. Splits the window if a response hits the 3,000-row limit.
    r = requests.get(API + path, headers=headers, timeout=90, params={
        "startDateRange": start.strftime("%Y-%m-%dT%H:%M:%SZ"), "endDateRange": end.strftime("%Y-%m-%dT%H:%M:%SZ")})
    r.raise_for_status()
    rows = r.json()
    pull.calls += 1
    time.sleep(0.25)
    if len(rows) >= 3000 and (end - start) > timedelta(days=1):
        mid = (start + (end - start) / 2).replace(microsecond=0)
        return pull(path, start, mid, headers) + pull(path, mid + timedelta(seconds=1), end, headers)
    return rows


pull.calls = 0


def pull_range(start, end, headers, label=""):
    raw = {"/games": [], "/games/teams": [], "/lines": []}
    for path, step in (("/games", 30), ("/games/teams", 15), ("/lines", 30)):
        cur = start
        while cur < end:
            nxt = min(cur + timedelta(days=step), end)
            raw[path].extend(pull(path, cur, nxt - timedelta(seconds=1), headers))
            cur = nxt
    if label:
        print(f"  {label}: {len(raw['/games']):,} games, {len(raw['/lines']):,} games with lines   ({pull.calls} calls so far)")
    return flatten_games(raw["/games"]), flatten_stats(raw["/games/teams"]), flatten_lines(raw["/lines"])


def full_pull(key):
    headers = {"Authorization": "Bearer " + key, "Accept": "application/json"}
    today = datetime.now(timezone.utc).replace(tzinfo=None)
    current = today.year + 1 if today.month >= 10 else today.year
    parts = []
    for season in range(2012, current + 1):
        end = min(datetime(season, 4, 20), today + timedelta(days=10))
        parts.append(pull_range(datetime(season - 1, 11, 1), end, headers, label=str(season)))
    return [pd.concat([p[i] for p in parts], ignore_index=True) for i in range(3)]


def save_zip(games, stats, lines):
    for df, name in zip((games, stats, lines), TABLES):
        df.to_parquet(name, index=False)
    with zipfile.ZipFile(DATA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for name in TABLES:
            z.write(name)


key = get_key()
if os.path.exists(DATA_ZIP):
    with zipfile.ZipFile(DATA_ZIP) as z:
        games, stats, lines = (pd.read_parquet(io.BytesIO(z.read(name))) for name in TABLES)
    source = "cbb_data.zip"
elif all(os.path.exists(name) for name in TABLES):
    games, stats, lines = (pd.read_parquet(name) for name in TABLES)
    source = "the three parquet files"
elif key:
    print("No data file found, so everything is being downloaded with your API key.")
    print("This takes about 12 minutes and roughly 360 API calls. Upload cbb_data.zip next time to skip it.")
    games, stats, lines = full_pull(key)
    save_zip(games, stats, lines)
    source = "a fresh download (saved as cbb_data.zip in the Files panel)"
else:
    raise SystemExit("No data found. Upload cbb_data.zip with the Files panel (folder icon on the left), "
                     "or switch on your CBBD_API_KEY secret for this notebook, then run it again.")

games = games.drop_duplicates("game_id", keep="last")
stats = stats.drop_duplicates(["game_id", "team_id"], keep="last")
lines = lines.drop_duplicates(["game_id", "provider"], keep="last")
print(f"Loaded {len(games):,} games, {len(stats):,} team box scores and {len(lines):,} line rows from {source}.")
print(f"Newest finished game: {games.loc[games['status'] == 'final', 'start_date'].max()[:10]}")

### Refresh the last three weeks

Finished seasons never change, so only recent dates are pulled again. Without an API key this step is skipped and the notebook uses the file as it is.

In [ ]:
def merge_new(old, new, keys):
    if len(new) == 0:
        return old
    keep = ~old.set_index(keys).index.isin(new.set_index(keys).index)
    return pd.concat([old[keep], new], ignore_index=True)


if key and not source.startswith("a fresh download"):
    try:
        today = datetime.now(timezone.utc).replace(tzinfo=None)
        newest = pd.to_datetime(games.loc[games["status"] == "final", "start_date"].max()).tz_localize(None)
        start = min(today - timedelta(days=21), newest - timedelta(days=3))
        headers = {"Authorization": "Bearer " + key, "Accept": "application/json"}
        new_games, new_stats, new_lines = pull_range(start, today + timedelta(days=10), headers)
        games = merge_new(games, new_games, ["game_id"])
        stats = merge_new(stats, new_stats, ["game_id", "team_id"])
        lines = merge_new(lines, new_lines, ["game_id", "provider"])
        save_zip(games, stats, lines)
        print(f"Refreshed {start:%b %d} onward with {pull.calls} API calls: {len(new_games):,} games and {len(new_lines):,} line rows.")
        print("cbb_data.zip in the Files panel now holds the fresh copy. Download it if you want to keep it.")
    except Exception as e:
        print(f"Refresh failed ({type(e).__name__}: {str(e)[:120]}). Continuing with the data as loaded.")
elif not key:
    print("No CBBD_API_KEY secret is switched on for this notebook, so the data was not refreshed.")
    print("To refresh: key icon in the left sidebar, then switch 'Notebook access' on for CBBD_API_KEY.")

---
# 2. Ratings

One pass through every game in date order. For each game the function **saves the pre-game numbers first, then updates with the result**, so no prediction ever sees its own outcome.

| Rating | What it measures |
|---|---|
| **Offense and defense** | Points per 100 possessions, adjusted for the opponent and for home court. A good night against a great defense counts for more. |
| **Shooting-luck-adjusted version** | The same, but with three-point makes replaced by what an average shooting night would produce. Three-point percentage swings a lot from game to game and mostly isn't repeatable. |
| **Tempo** | Possessions per game, to turn efficiency into points |
| **Elo** | Overall strength from wins and margins |

Three details matter in college basketball:

- **Early games move ratings more.** A team's first few games get extra weight, then it settles.
- **Between seasons, teams regress toward their conference's level**, not the national average. A mid-major does not drift toward the Big Ten.
- **Teams outside Division I start far below average.**

The settings were tuned on seasons through 2021 only, using the model's own prediction error.

In [ ]:
# Data prep
def prepare_games(games):
    g = games.copy()
    g["start_date"] = pd.to_datetime(g["start_date"], utc=True)
    g["date"] = (g["start_date"] - pd.Timedelta(hours=10)).dt.tz_localize(None).dt.normalize()   # US game day
    g = g[g["home_team_id"].notna() & g["away_team_id"].notna()]
    g["played"] = (g["status"] == "final") & g["home_points"].notna() & g["away_points"].notna()
    g = g[g["played"] | (g["status"] == "scheduled")]
    g["neutral"] = g["neutral_site"].fillna(False).astype(int)
    g["margin"] = g["home_points"] - g["away_points"]
    g["home_win"] = np.where(g["played"], (g["margin"] > 0).astype(float), np.nan)
    g["home_d1"] = g["home_conference"].notna().astype(int)
    g["away_d1"] = g["away_conference"].notna().astype(int)
    return g.sort_values(["start_date", "game_id"]).reset_index(drop=True)


def pick_market(lines, provider_order):
    """One moneyline per game: the first provider in provider_order that has both sides.
    Also keeps that provider's spread, and flags prices that look broken."""
    l = lines[lines["home_moneyline"].notna() & lines["away_moneyline"].notna()].copy()
    l = l[(l["home_moneyline"].abs() >= 100) & (l["away_moneyline"].abs() >= 100)]
    rank = {p: i for i, p in enumerate(provider_order)}
    l = l[l["provider"].isin(rank)]
    l["rank"] = l["provider"].map(rank)
    l = l.sort_values(["game_id", "rank"]).drop_duplicates("game_id")
    ph, pa = implied_prob(l["home_moneyline"]), implied_prob(l["away_moneyline"])
    l["overround"] = ph + pa - 1
    l["mkt_home_prob"] = ph / (ph + pa)
    return l[["game_id", "provider", "home_moneyline", "away_moneyline", "spread", "over_under",
              "spread_open", "overround", "mkt_home_prob"]]


# Ratings: one pass through the games in date order
DEFAULTS = dict(
    hca_eff=2.3,          # home edge, points per 100 possessions on each side of the ball
    k_floor=0.07,         # long-run weight of the newest game
    prior_games=4.0,      # a preseason rating is worth this many games
    carry=0.62,           # share of last season's rating kept
    nond1_off=-13.0,      # starting rating for teams outside Division I
    cap=45.0,             # largest single-game surprise allowed, per 100 possessions
    tempo_k=0.10,
    elo_k=28.0, elo_hca=75.0, elo_carry=0.70, elo_nond1=1100.0,
    luck_shrink=0.5,      # how much of 3-point shooting luck is removed in the adjusted ratings
    conf_pull=1.0,        # between seasons, teams regress toward their conference average (1) or the national average (0)
)


def build_ratings(g, stats, params=None):
    """g: prepared games. stats: team box scores (game_id, team_id, possessions, fg3a, fg3m, points).
    Returns a DataFrame of pre-game features aligned with g, and the final rating state."""
    P = dict(DEFAULTS)
    if params:
        P.update(params)
    s = stats.dropna(subset=["possessions"])
    box = {(a, b): (c, d, e) for a, b, c, d, e in zip(s["game_id"], s["team_id"], s["possessions"], s["fg3a"], s["fg3m"])}

    off, dfn, off2, dfn2, tempo, elo = {}, {}, {}, {}, {}, {}
    n_games, last_season, last_date, conf_of = {}, {}, {}, {}
    league = {"eff": 102.0, "tempo": 68.0, "p3": 0.345}
    conf_mean, current_season = {}, None
    rows = []

    for r in g.itertuples(index=False):
        h, a, season = r.home_team_id, r.away_team_id, r.season
        if season != current_season:
            # New season: remember each conference's average rating from the season that just ended.
            current_season, conf_mean = season, {}
            for book_name, book in (("off", off), ("dfn", dfn), ("off2", off2), ("dfn2", dfn2), ("elo", elo)):
                sums = {}
                for t, c in conf_of.items():
                    if c is not None and t in book:
                        tot = sums.setdefault(c, [0.0, 0])
                        tot[0] += book[t]; tot[1] += 1
                conf_mean[book_name] = {c: v[0] / v[1] for c, v in sums.items()}
        for t, d1, conf in ((h, r.home_d1, r.home_conference), (a, r.away_d1, r.away_conference)):
            if t not in off:
                base = 0.0 if d1 else P["nond1_off"]
                # a team new to the data starts at its conference's level if we know it
                o0 = conf_mean.get("off", {}).get(conf, base) if d1 else base
                d0 = conf_mean.get("dfn", {}).get(conf, -base) if d1 else -base
                off[t], dfn[t], off2[t], dfn2[t], tempo[t] = o0, d0, o0, d0, 0.0
                elo[t] = conf_mean.get("elo", {}).get(conf, 1500.0) if d1 else P["elo_nond1"]
                n_games[t] = 0
            elif last_season[t] != season:
                w = P["conf_pull"]
                for name, book in (("off", off), ("dfn", dfn), ("off2", off2), ("dfn2", dfn2)):
                    target = w * conf_mean.get(name, {}).get(conf, 0.0)
                    book[t] = target + P["carry"] * (book[t] - target)
                tempo[t] *= 0.75
                if d1:
                    target = 1500.0 + w * (conf_mean.get("elo", {}).get(conf, 1500.0) - 1500.0)
                    elo[t] = target + P["elo_carry"] * (elo[t] - target)
                n_games[t] = 0
            last_season[t] = season
            conf_of[t] = conf if d1 else None

        hca = 0.0 if r.neutral else P["hca_eff"]
        exp_poss = league["tempo"] + tempo[h] + tempo[a]
        net = (off[h] - dfn[h]) - (off[a] - dfn[a])           # per 100 possessions; dfn positive = bad defense
        net2 = (off2[h] - dfn2[h]) - (off2[a] - dfn2[a])
        rest_h = min((r.date - last_date[h]).days, 10) if h in last_date else 10
        rest_a = min((r.date - last_date[a]).days, 10) if a in last_date else 10

        # ---- 1) pre-game features ----
        rows.append((
            (net + 2 * hca) * exp_poss / 100,                 # rating-based margin, raw
            (net2 + 2 * hca) * exp_poss / 100,                # rating-based margin, shooting-luck adjusted
            (2 * league["eff"] + off[h] + dfn[a] + off[a] + dfn[h]) * exp_poss / 100,     # rating-based total
            elo[h] - elo[a] + (0 if r.neutral else P["elo_hca"]),
            exp_poss, off[h], dfn[h], off[a], dfn[a],
            n_games[h], n_games[a], rest_h, rest_a,
        ))
        if not r.played:
            continue

        # ---- 2) update with the result ----
        bh, ba = box.get((r.game_id, h)), box.get((r.game_id, a))
        if bh is not None and ba is not None and bh[0] > 40 and ba[0] > 40:
            poss = (bh[0] + ba[0]) / 2
            lucky_h = 3 * (bh[2] - bh[1] * league["p3"]) if bh[1] == bh[1] and bh[2] == bh[2] else 0.0
            lucky_a = 3 * (ba[2] - ba[1] * league["p3"]) if ba[1] == ba[1] and ba[2] == ba[2] else 0.0
            if bh[1] == bh[1] and bh[1] + ba[1] > 0 and bh[2] == bh[2] and ba[2] == ba[2]:
                league["p3"] += 0.0005 * ((bh[2] + ba[2]) / (bh[1] + ba[1]) - league["p3"])
        else:
            poss, lucky_h, lucky_a = exp_poss, 0.0, 0.0
        eh, ea = 100 * r.home_points / poss, 100 * r.away_points / poss
        eh2 = 100 * (r.home_points - P["luck_shrink"] * lucky_h) / poss
        ea2 = 100 * (r.away_points - P["luck_shrink"] * lucky_a) / poss

        kh = max(P["k_floor"], 1 / (n_games[h] + P["prior_games"]))
        ka = max(P["k_floor"], 1 / (n_games[a] + P["prior_games"]))
        for o, d, obs_h, obs_a in ((off, dfn, eh, ea), (off2, dfn2, eh2, ea2)):
            err_h = np.clip(obs_h - (league["eff"] + o[h] + d[a] + hca), -P["cap"], P["cap"])
            err_a = np.clip(obs_a - (league["eff"] + o[a] + d[h] - hca), -P["cap"], P["cap"])
            o[h] += kh * err_h; d[a] += ka * err_h
            o[a] += ka * err_a; d[h] += kh * err_a
        if r.home_d1 and r.away_d1:
            league["eff"] += 0.0008 * ((eh + ea) / 2 - league["eff"])
            league["tempo"] += 0.0008 * (poss - league["tempo"])
        terr = poss - exp_poss
        tempo[h] += P["tempo_k"] * terr; tempo[a] += P["tempo_k"] * terr

        gap = elo[h] - elo[a] + (0 if r.neutral else P["elo_hca"])
        exp_h = 1 / (1 + 10 ** (-gap / 400))
        win_h = 1.0 if r.margin > 0 else 0.0
        wgap = gap if r.margin > 0 else -gap
        mult = np.log(abs(r.margin) + 1) * 2.2 / (wgap * 0.001 + 2.2)
        ch = P["elo_k"] * mult * (win_h - exp_h)
        elo[h] += ch; elo[a] -= ch

        n_games[h] += 1; n_games[a] += 1
        last_date[h] = last_date[a] = r.date

    cols = ["rate_margin", "rate_margin_adj", "rate_total", "elo_gap", "exp_poss",
            "home_off", "home_def", "away_off", "away_def", "home_n", "away_n", "home_rest", "away_rest"]
    feats = pd.DataFrame(rows, columns=cols, index=g.index)
    state = dict(off=off, dfn=dfn, off2=off2, dfn2=dfn2, tempo=tempo, elo=elo, league=league, n_games=n_games)
    return feats, state


def add_features(g):
    g = g.copy()
    g["home_field"] = 1 - g["neutral"]
    g["rest_diff"] = (g["home_rest"] - g["away_rest"]).clip(-5, 5)
    g["min_n"] = np.minimum(g["home_n"], g["away_n"]).clip(upper=15)
    g["early"] = (15 - g["min_n"]) / 15                       # 1 at season start, 0 after 15 games
    g["rate_x_early"] = g["rate_margin"] * g["early"]
    g["elo_pts"] = g["elo_gap"] / 28.0
    g["mismatch"] = g["home_d1"] - g["away_d1"]
    return g


FEATURES = ["rate_margin", "rate_margin_adj", "elo_pts", "home_field", "rest_diff", "rate_x_early", "mismatch"]

In [ ]:
# Settings tuned on seasons through 2021 (see section 5 for why that cut-off matters).
PARAMS = {"hca_eff": 1.4, "k_floor": 0.03, "prior_games": 5, "carry": 0.9, "nond1_off": -16, "cap": 30, "tempo_k": 0.03,
          "elo_k": 24, "elo_hca": 60, "elo_carry": 0.5, "luck_shrink": 1.0}

started = time.time()
g = prepare_games(games)
# old "scheduled" rows that never got a result are not upcoming games
now = pd.Timestamp.now(tz="UTC")
g = g[g["played"] | (g["start_date"] >= now - pd.Timedelta(hours=6))].reset_index(drop=True)
features, state = build_ratings(g, stats, PARAMS)
g = add_features(g.join(features))
LAST_SEASON = int(g["season"].max())
print(f"Rated {g['played'].sum():,} finished games in {time.time() - started:.0f}s. Upcoming games in the data: {(~g['played']).sum():,}.")

### Do the ratings look right?

The current top 15 by net rating (offense plus defense; higher is better). If this looks like the top of the polls, the ratings are working.

Early in a season the numbers lean heavily on last season, and the gaps between teams are stretched. Read them as a ranking, not as exact points.

In [ ]:
names = pd.concat([g[["home_team_id", "home_team", "home_conference"]].set_axis(["id", "team", "conf"], axis=1),
                   g[["away_team_id", "away_team", "away_conference"]].set_axis(["id", "team", "conf"], axis=1)]).drop_duplicates("id", keep="last").set_index("id")
active = [t for t, n in state["n_games"].items() if n > 0 and pd.notna(names.loc[t, "conf"])]
ratings = pd.DataFrame({
    "team": [names.loc[t, "team"] for t in active], "conference": [names.loc[t, "conf"] for t in active],
    "games": [state["n_games"][t] for t in active],
    "offense": [state["off"][t] for t in active], "defense": [-state["dfn"][t] for t in active],
    "tempo": [state["league"]["tempo"] + state["tempo"][t] for t in active], "elo": [state["elo"][t] for t in active],
})
ratings["net"] = ratings["offense"] + ratings["defense"]
ratings = ratings.sort_values("net", ascending=False).reset_index(drop=True)
ratings.index += 1
ratings.round(1).to_csv("cbb_team_ratings.csv", index_label="rank")
print(f"{len(ratings)} Division I teams have played this season. 'defense' is flipped so higher is better.")
ratings.head(15).round(1)

---
# 3. Moneylines, and a trap

## One price per game

Several providers list each game. The model uses one per game, in a fixed order that puts real sportsbooks first:

`ESPN BET → DraftKings → Bovada → teamrankings → numberfire → consensus`

The last three are odds-listing sites, not books, and they are the only source before 2023. Their prices carry less vig than a typical sportsbook, which flatters the older results. Section 6 shows results by provider so you can see the difference.

## The trap: swapped lines

In this data, some neutral-site games have the moneyline attached to the **wrong team**: the favorite is listed as the underdog. A model that knows the teams will "disagree" with that line, bet the listed underdog at +1200, and win, because that team was really the favorite.

The first version of this model showed a 40% return for exactly that reason. It was not an edge. It was a data error, and you could never have placed those bets.

So every price has to pass these checks before it is used for anything:

1. **Sensible vig**: the two sides must add up to between 100.5% and 112%.
2. **Agrees with the spread**: the moneyline and the same provider's spread must point the same way.
3. **Providers agree**: if two providers disagree by 15 points of probability, the game is dropped.
4. **No giant disagreements**: if the model and the market are 25 points apart, that is treated as a data problem or as news the model doesn't have, never as a bet.

In [ ]:
PROVIDERS = ["ESPN BET", "Draft Kings", "Bovada", "teamrankings", "numberfire", "consensus"]
SPORTSBOOKS = ["ESPN BET", "Draft Kings", "Bovada"]
FIRST_TRAIN, FIRST_TEST, FIRST_CAL = 2013, 2015, 2017
VALID = (2017, 2021)
HOLDOUT_START = 2022

market = pick_market(lines, PROVIDERS)
both = lines[lines["home_moneyline"].notna() & lines["away_moneyline"].notna()
             & (lines["home_moneyline"].abs() >= 100) & (lines["away_moneyline"].abs() >= 100)].copy()
ph, pa = implied_prob(both["home_moneyline"]), implied_prob(both["away_moneyline"])
both["fair"] = ph / (ph + pa)
provider_range = both.groupby("game_id")["fair"].agg(lambda x: x.max() - x.min()).rename("provider_range")
g = g.merge(market, on="game_id", how="left").merge(provider_range, on="game_id", how="left")

coverage = g[g["mkt_home_prob"].notna()].groupby(["season", "provider"]).size().unstack().fillna(0).astype(int)
print("GAMES WITH A MONEYLINE, BY SEASON AND THE PROVIDER USED")
coverage

---
# 4. The model

## 4a. The model's own probability

A logistic regression predicts the home team's chance to win from the ratings: the efficiency-based margin (raw and luck-adjusted), Elo, home court, rest, how early in the season it is, and whether one team is outside Division I.

It is trained **walk-forward**: each season is predicted by a model that has only seen earlier seasons.

In [ ]:
def walk_forward_prob(g, feature_cols, first_train, first_test, last_test, train_mask):
    p = pd.Series(np.nan, index=g.index)
    for season in range(first_test, last_test + 1):
        train = g[g["played"] & train_mask & g["season"].between(first_train, season - 1)]
        test = g[g["season"] == season]
        if len(test) == 0 or len(train) < 500:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)).fit(train[feature_cols], train["home_win"])
        p.loc[test.index] = clf.predict_proba(test[feature_cols])[:, 1]
    return p


d1 = (g["home_d1"] == 1) & (g["away_d1"] == 1)
g["p_model"] = walk_forward_prob(g, FEATURES, FIRST_TRAIN, FIRST_TEST, LAST_SEASON, d1)

# Price checks (section 3)
spread_prob = norm.cdf(-g["spread"] / 10.5)
g["price_ok"] = (g["mkt_home_prob"].notna() & g["overround"].between(0.005, 0.12)
                 & ((g["mkt_home_prob"] - spread_prob).abs().fillna(0) < 0.15) & (g["provider_range"].fillna(0) < 0.15))
gap = (g["p_model"] - g["mkt_home_prob"]).abs()
gap_if_swapped = (g["p_model"] - (1 - g["mkt_home_prob"])).abs()
g["suspect"] = ((gap >= 0.25) | ((g["neutral"] == 1) & (gap > 0.15) & (gap_if_swapped < gap - 0.10))).fillna(False)
g["usable"] = g["price_ok"] & ~g["suspect"]

n_ml = int(g["mkt_home_prob"].notna().sum())
print(f"Moneylines found:                    {n_ml:,}")
print(f"Dropped by the price checks:         {int((g['mkt_home_prob'].notna() & ~g['price_ok']).sum()):,}")
print(f"Dropped as suspected swaps or news:  {int((g['price_ok'] & g['suspect']).sum()):,}")
print(f"Usable:                              {int(g['usable'].sum()):,}")

swapped = g[g["price_ok"] & g["suspect"] & g["played"]]
liked_home = swapped["p_model"] > swapped["mkt_home_prob"]
print(f"\nIn the dropped games, the side the model liked was priced at {np.where(liked_home, swapped['mkt_home_prob'], 1 - swapped['mkt_home_prob']).mean():.0%} "
      f"on average and won {np.where(liked_home, swapped['home_win'], 1 - swapped['home_win']).mean():.0%} of the time.")
print("Those would have looked like huge winners. They are data errors, which is why they are removed.")
g.loc[~g["usable"], ["mkt_home_prob", "home_moneyline", "away_moneyline"]] = np.nan

## 4b. The market by itself

Before blending anything, look at what the market does alone. For every usable moneyline, group both sides by the price's break-even probability and see how each group actually paid.

In [ ]:
def candidates(d, prob_col):
    # Both sides of every moneyline, with a probability, the price and the result.
    d = d[d[prob_col].notna() & d["home_moneyline"].notna()]
    parts = []
    for side, p, price, other, won, team, mkt in (
        ("home", d[prob_col], d["home_moneyline"], d["away_moneyline"], d["home_win"], d["home_team"], d["mkt_home_prob"]),
        ("away", 1 - d[prob_col], d["away_moneyline"], d["home_moneyline"], 1 - d["home_win"], d["away_team"], 1 - d["mkt_home_prob"]),
    ):
        parts.append(pd.DataFrame({
            "game_id": d["game_id"], "season": d["season"], "date": d["date"], "start_date": d["start_date"], "played": d["played"],
            "matchup": d["away_team"] + " at " + d["home_team"], "provider": d["provider"],
            "side": side, "pick": team, "p": p, "price": price, "won": won, "market_prob": mkt,
            "model_prob": np.where(side == "home", d["p_model"], 1 - d["p_model"])}))
    c = pd.concat(parts, ignore_index=True)
    c["break_even"] = implied_prob(c["price"])
    c["decimal"] = american_to_decimal(c["price"])
    c["edge"] = c["p"] - c["break_even"]
    c["ev"] = c["p"] * (c["decimal"] - 1) - (1 - c["p"])
    c["profit"] = np.where(c["won"].isna(), np.nan, np.where(c["won"] == 1, c["decimal"] - 1, -1.0))
    return c


def roi_range(b, n_boot=2000, seed=0):
    profit = b["profit"].dropna().to_numpy()
    boot = profit[np.random.default_rng(seed).integers(0, len(profit), size=(n_boot, len(profit)))].mean(axis=1)
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def summarize(b):
    b = b[b["profit"].notna()]
    if len(b) == 0:
        return pd.Series({"bets": 0})
    low, high = roi_range(b)
    return pd.Series({"bets": len(b), "hit_rate_%": round(b["won"].mean() * 100, 1), "needed_%": round(b["break_even"].mean() * 100, 1),
                      "avg_decimal_price": round(b["decimal"].mean(), 2), "roi_%": round(b["profit"].mean() * 100, 2),
                      "roi_95%_range": f"{low * 100:+.1f} to {high * 100:+.1f}", "units": round(b["profit"].sum(), 1)})


def summarize_by(b, key):
    return pd.DataFrame({k: summarize(v) for k, v in b.groupby(key, observed=True)}).T.rename_axis(key)


PRICE_BINS = [0, 0.15, 0.3, 0.5, 0.65, 0.8, 0.9, 1.0]
PRICE_LABELS = ["Big underdog (<15%)", "Underdog (15–30%)", "Slight underdog (30–50%)", "Slight favorite (50–65%)",
                "Favorite (65–80%)", "Big favorite (80–90%)", "Huge favorite (90%+)"]
every_side = candidates(g[g["played"] & g["mkt_home_prob"].notna()].assign(p_mkt=lambda d: d["mkt_home_prob"]), "p_mkt")
every_side["price_group"] = pd.cut(every_side["break_even"], PRICE_BINS, labels=PRICE_LABELS)
blind = summarize_by(every_side, "price_group")
print(f"BETTING EVERY SIDE BLINDLY, {int(every_side['season'].min())}–{LAST_SEASON}: what each price group returned")
blind

**This is the most important table for a moneyline bettor.** It is called the favorite–longshot bias:

- **Underdogs are badly overpriced.** Big underdogs lose a large share of every dollar bet. The vig is not spread evenly; most of it sits on the longshots.
- **Favorites are close to fair.** Big favorites lose only a little.

Two things follow. First, a high hit rate comes for free: betting every huge favorite wins more than 90% of the time, and it still loses a little money. Second, any edge has to come from the favorite side of the board, because that is the only place the price is nearly fair.

## 4c. The final probability

The final number combines three sources in one logistic regression, trained walk-forward on earlier seasons only:

`final = market moneyline + (model − market) + (spread − market)`

- **The market moneyline** is the anchor. Its weight comes out above 1, which is the favorite–longshot bias: favorites win a bit more often than their moneyline says.
- **The model's disagreement** gets a small weight.
- **The spread's disagreement** matters too. The point-spread market takes far more money than the moneyline market, so when a game's spread implies a different win probability than its moneyline, the spread is usually closer to the truth.

In [ ]:
# Win probability implied by the spread (scaled by the total: higher-scoring games are more random per point)
g["spread_scaled"] = -g["spread"] / np.sqrt(g["over_under"].clip(100, 190).fillna(140) / 140)
g["p_spread"] = np.nan
for season in range(FIRST_TEST, LAST_SEASON + 1):
    train = g[g["played"] & g["spread_scaled"].notna() & g["season"].between(FIRST_TRAIN, season - 1)]
    test = g[(g["season"] == season) & g["spread_scaled"].notna()]
    if len(train) > 1500 and len(test):
        g.loc[test.index, "p_spread"] = LogisticRegression(C=1e6).fit(train[["spread_scaled"]], train["home_win"]).predict_proba(test[["spread_scaled"]])[:, 1]

g["market_logit"] = logit(g["mkt_home_prob"])
g["model_gap"] = logit(g["p_model"]) - g["market_logit"]
g["spread_gap"] = logit(g["p_spread"]) - g["market_logit"]


def blend(g, cols):
    out, weights, last_fit = pd.Series(np.nan, index=g.index), [], None
    have = g["mkt_home_prob"].notna() & g[cols].notna().all(axis=1)
    for season in range(FIRST_CAL, LAST_SEASON + 1):
        train, test = g[have & g["played"] & (g["season"] < season)], g[have & (g["season"] == season)]
        if len(train) < 1500 or len(test) == 0:
            continue
        last_fit = LogisticRegression(C=1e6, max_iter=3000).fit(train[cols], train["home_win"])
        out.loc[test.index] = last_fit.predict_proba(test[cols])[:, 1]
        weights.append({"season": season, **dict(zip(cols, last_fit.coef_[0])), "intercept": last_fit.intercept_[0], "games_trained_on": len(train)})
    return out, pd.DataFrame(weights).set_index("season"), last_fit


p_full, weights, fit_full = blend(g, ["market_logit", "model_gap", "spread_gap"])
p_simple, _, fit_simple = blend(g, ["market_logit", "model_gap"])        # used when a game has no spread
g["p_final"] = p_full.fillna(p_simple)
print("WEIGHTS LEARNED FOR EACH SEASON (from earlier seasons only)")
weights.rename(columns={"market_logit": "market", "model_gap": "model_disagreement", "spread_gap": "spread_disagreement"}).round(3)

---
# 5. The test

## How it was kept honest

Every choice in this notebook, including the rating settings, the price checks, the blend and the betting rule, was made using seasons **through 2021 only**. Seasons from **2022 onward** were set aside and examined once, after everything was fixed. That holdout is the number to believe.

| Period | Seasons | Used for |
|---|---|---|
| Warm-up | 2012–2014 | Getting ratings and the first models started |
| Validation | 2017–2021 | Choosing settings and the betting rule |
| **Holdout** | **2022 onward** | **Judged once. Nothing was tuned on it.** |

**The betting rule, fixed in advance:** bet 1 unit on a side whenever the final probability beats the break-even probability at its price. A stricter version (edge of 1 point or more) is shown beside it.

## 5a. Probability accuracy

Log loss and Brier score both measure how good the probabilities are; lower is better. The market's vig-free moneyline is the benchmark.

In [ ]:
def period(lo, hi):
    return g[g["played"] & g["season"].between(lo, hi) & g["mkt_home_prob"].notna() & g["p_final"].notna()]


valid_games, hold_games = period(*VALID), period(HOLDOUT_START, LAST_SEASON)
accuracy = {}
for label, d in ((f"Validation {VALID[0]}–{VALID[1]}", valid_games), (f"Holdout {HOLDOUT_START}–{LAST_SEASON}", hold_games)):
    for name, col in (("Market moneyline", "mkt_home_prob"), ("Model alone", "p_model"), ("Final (market + model + spread)", "p_final")):
        accuracy[(label, name)] = {"games": len(d), "log_loss": logloss(d[col], d["home_win"]), "brier": brier(d[col], d["home_win"])}
accuracy = pd.DataFrame(accuracy).T
accuracy["games"] = accuracy["games"].astype(int)
accuracy.round(4)

**How to read it:** the model alone is a little worse than the market, as expected. The final blend is **better than the market** in both periods, including the holdout it never saw. So the final probabilities are a real improvement on the raw moneyline.

Better probabilities are necessary for a betting edge, but not sufficient. The improvement also has to be bigger than the vig.

## 5b. Is the final probability calibrated?

In [ ]:
cal = hold_games.assign(bucket=pd.cut(hold_games["p_final"], np.arange(0, 1.01, 0.1)))
calib = cal.groupby("bucket", observed=True).agg(games=("home_win", "size"), final_prob=("p_final", "mean"),
                                                 market_prob=("mkt_home_prob", "mean"), actual_win_rate=("home_win", "mean"))
fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
ax.scatter(calib["market_prob"], calib["actual_win_rate"], s=60, marker="x", color="tab:gray", label="Market moneyline (vig removed)")
ax.scatter(calib["final_prob"], calib["actual_win_rate"], s=calib["games"] / 25, alpha=0.8, label="Final probability")
ax.set_xlabel("Predicted home win probability"); ax.set_ylabel("How often the home team won")
ax.set_title(f"Holdout calibration, {HOLDOUT_START}–{LAST_SEASON}")
ax.legend(loc="upper left", fontsize=9); plt.show()
calib.round(3)

## 5c. Betting backtest

The reference rows show what happens with no model at all.

In [ ]:
def backtest(d):
    c = candidates(d, "p_final")
    rows = {
        "PRIMARY RULE: any positive edge": summarize(c[c["edge"] >= 0]),
        "Stricter: edge of 1 point or more": summarize(c[c["edge"] >= 0.01]),
        "Reference: every big favorite (80%+)": summarize(c[c["break_even"] >= 0.8]),
        "Reference: every underdog (<30%)": summarize(c[c["break_even"] < 0.3]),
        "Reference: every side of every game": summarize(c),
    }
    return c, pd.DataFrame(rows).T


valid_c, valid_table = backtest(valid_games)
hold_c, hold_table = backtest(hold_games)
print(f"VALIDATION {VALID[0]}–{VALID[1]} (the rule was chosen here, so this flatters it slightly)")
display(valid_table)
print(f"HOLDOUT {HOLDOUT_START}–{LAST_SEASON} (never used for any choice)")
display(hold_table)

**How to read it**

- Compare `hit_rate_%` to `needed_%`. The gap between them is the whole story.
- `roi_95%_range` is the uncertainty. If it includes zero, the result cannot be told apart from break-even.
- Compare the primary rule with "every side of every game". That gap is what the model is worth.

## 5d. The primary rule, season by season and by sportsbook

In [ ]:
primary = hold_c[hold_c["edge"] >= 0].copy()
print("HOLDOUT, PRIMARY RULE BY SEASON")
display(summarize_by(primary, "season"))
print("HOLDOUT, PRIMARY RULE BY PROVIDER (ESPN BET, Draft Kings and Bovada are real sportsbooks)")
display(summarize_by(primary, "provider"))
books = primary[primary["provider"].isin(SPORTSBOOKS)]
print("HOLDOUT, PRIMARY RULE AT REAL SPORTSBOOK PRICES ONLY")
display(summarize(books).to_frame().T)

fig, ax = plt.subplots()
line = primary[primary["profit"].notna()].sort_values("start_date")
ax.plot(line["date"], line["profit"].cumsum())
ax.axhline(0, color="k", linewidth=1)
ax.set_xlabel("Date"); ax.set_ylabel("Cumulative profit (units, 1 unit per bet)")
ax.set_title(f"Holdout: primary rule, {len(line):,} bets")
plt.show()

## 5e. Where the bets land

The same holdout bets, grouped by price. This shows which part of the board the rule plays in.

In [ ]:
primary["price_group"] = pd.cut(primary["break_even"], PRICE_BINS, labels=PRICE_LABELS)
by_price = summarize_by(primary, "price_group")
validation_primary = valid_c[valid_c["edge"] >= 0].copy()
validation_primary["price_group"] = pd.cut(validation_primary["break_even"], PRICE_BINS, labels=PRICE_LABELS)
print("HOLDOUT"); display(by_price)
print("VALIDATION, for comparison"); display(summarize_by(validation_primary, "price_group"))

Be careful with this table. It slices one result seven ways, and slices always produce a few good-looking rows by chance. A row only deserves attention if it is positive in **both** periods, and even then it is a lead to track live, not a strategy.

One specific warning about the slight-underdog rows: those bets exist because the moneyline and the spread disagreed. In historical data that often means the two were recorded at different times, which you could not have bet. Treat that row with extra suspicion.

## 5f. The verdict

Written from the tables above each time the notebook runs.

In [ ]:
p_row, s_row, all_row = hold_table.iloc[0], hold_table.iloc[1], hold_table.iloc[4]
low, high = roi_range(primary)
book_roi = books["profit"].mean() * 100 if len(books) else float("nan")
mkt_ll = accuracy.loc[(f"Holdout {HOLDOUT_START}–{LAST_SEASON}", "Market moneyline"), "log_loss"]
fin_ll = accuracy.loc[(f"Holdout {HOLDOUT_START}–{LAST_SEASON}", "Final (market + model + spread)"), "log_loss"]

print(f"VERDICT ON THE HOLDOUT, {HOLDOUT_START}–{LAST_SEASON}")
print()
print(f"1. Probabilities. The final blend beats the market moneyline ({fin_ll:.4f} vs {mkt_ll:.4f} log loss). That part works.")
print()
print(f"2. Betting. The primary rule placed {int(p_row['bets']):,} bets and hit {p_row['hit_rate_%']:.1f}%. It needed {p_row['needed_%']:.1f}% to break even.")
print(f"   Return: {p_row['roi_%']:+.2f}% per unit bet, with a 95% range of {low * 100:+.1f}% to {high * 100:+.1f}%.")
print(f"   At real sportsbook prices only: {book_roi:+.2f}% on {len(books):,} bets.")
print(f"   Betting every side with no model returned {all_row['roi_%']:+.2f}%.")
print()
if low > 0:
    print("BOTTOM LINE: the rule shows a profit whose range stays above zero. Confirm it with live tracking before trusting it.")
elif p_row["roi_%"] > -1.0:
    print("BOTTOM LINE: the model breaks even. It removes almost all of the sportsbook's edge, and it does not create one of its own.")
    print("It is not a proven way to make money. The range includes zero, so the true return could be slightly positive or slightly negative.")
else:
    print("BOTTOM LINE: the model loses less than blind betting, but it does not beat the closing moneyline.")

---
# 6. Upcoming games

Every upcoming game with a usable moneyline. For each game the table shows the side with the better edge.

- **market_prob**: the vig-free probability from the moneyline
- **model_prob**: the model alone
- **final_prob**: market + model + spread. This is the headline number.
- **edge**: `final_prob` minus the break-even probability at the listed price
- **lean**: `LEAN` when the edge is positive, which is the primary rule from section 5

`LEAN` is where the model sees value. Section 5 says that rule has broken even, so these are opinions, not recommendations. Prices are whatever the data source last recorded, so check the current line before doing anything with them.

In [ ]:
def score_upcoming(g):
    # Upcoming games use the most recently trained blend.
    up = g[~g["played"] & g["usable"] & g["p_model"].notna()].copy()
    if len(up) == 0:
        return up
    full = up[["market_logit", "model_gap", "spread_gap"]].notna().all(axis=1)
    up["p_final"] = np.nan
    if full.any():
        up.loc[full, "p_final"] = fit_full.predict_proba(up.loc[full, ["market_logit", "model_gap", "spread_gap"]])[:, 1]
    if (~full).any():
        up.loc[~full, "p_final"] = fit_simple.predict_proba(up.loc[~full, ["market_logit", "model_gap"]])[:, 1]
    return up


upcoming = score_upcoming(g)
if len(upcoming) == 0:
    picks = pd.DataFrame()
    print("No upcoming games with a usable moneyline right now. Refresh the data closer to game day.")
else:
    c = candidates(upcoming, "p_final").sort_values("edge", ascending=False).drop_duplicates("game_id")
    c["lean"] = np.where(c["edge"] >= 0, "LEAN", "-")
    c["tipoff_utc"] = c["start_date"].dt.strftime("%a %b %d %H:%M")
    picks = c.sort_values(["start_date", "matchup"])[["tipoff_utc", "matchup", "pick", "price", "provider", "market_prob", "model_prob", "p", "edge", "ev", "lean"]]
    picks = picks.rename(columns={"p": "final_prob"})
    picks.to_csv("cbb_upcoming_picks.csv", index=False)
    show = picks.copy()
    for col in ("market_prob", "model_prob", "final_prob", "edge", "ev"):
        show[col] = (show[col] * 100).round(1)
    show["price"] = show["price"].map("{:+.0f}".format)
    print(f"{len(picks)} upcoming games with a usable moneyline; {(picks['lean'] == 'LEAN').sum()} leans. Probabilities, edge and EV are in percent. Times are UTC.")
    display(show.set_index(["tipoff_utc", "matchup"]).head(60))

---
# 7. Save for the website

| File | Contents |
|---|---|
| `cbb_team_ratings.csv` | Current offense, defense, tempo and Elo for every Division I team |
| `cbb_upcoming_picks.csv` | The upcoming-games table |
| `cbb_backtest_bets.csv` | Every bet the primary rule placed since 2017, for the site's performance page |
| `cbb_track_record.csv` | Holdout results by price group |
| `cbb_model.json` | The blend weights and settings, small enough to read by eye |

In [ ]:
all_c = pd.concat([valid_c.assign(period="validation"), hold_c.assign(period="holdout")])
all_c = all_c[all_c["edge"] >= 0]          # the bets the primary rule placed
all_c[["period", "game_id", "season", "date", "matchup", "provider", "side", "pick", "price", "market_prob", "model_prob", "p",
       "break_even", "edge", "ev", "won", "profit"]].rename(columns={"p": "final_prob"}).to_csv("cbb_backtest_bets.csv", index=False)
by_price.reset_index().to_csv("cbb_track_record.csv", index=False)
bundle = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "trained_through": str(g.loc[g["played"], "date"].max().date()),
    "rating_settings": PARAMS, "model_inputs": FEATURES, "providers_in_order": PROVIDERS,
    "blend_weights": {"inputs": ["market_logit", "model_gap", "spread_gap"], "coef": [float(x) for x in fit_full.coef_[0]], "intercept": float(fit_full.intercept_[0])},
    "blend_weights_no_spread": {"inputs": ["market_logit", "model_gap"], "coef": [float(x) for x in fit_simple.coef_[0]], "intercept": float(fit_simple.intercept_[0])},
    "holdout": {"seasons": [HOLDOUT_START, LAST_SEASON], "primary_rule_bets": int(p_row["bets"]), "primary_rule_hit_rate_pct": float(p_row["hit_rate_%"]),
                "primary_rule_needed_pct": float(p_row["needed_%"]), "primary_rule_roi_pct": float(p_row["roi_%"])},
}
with open("cbb_model.json", "w") as f:
    json.dump(bundle, f, indent=2)
for name in ("cbb_team_ratings.csv", "cbb_upcoming_picks.csv", "cbb_backtest_bets.csv", "cbb_track_record.csv", "cbb_model.json"):
    if os.path.exists(name):
        print(f"  saved {name:28s} {os.path.getsize(name) / 1024:8.0f} KB")

---
# What this model is, and isn't

**What it does well.** Its probabilities are better than the raw moneyline on seasons it never saw. It turns blind betting, which loses heavily, into roughly break-even betting. It also avoids the part of the board where the sportsbook's edge is largest.

**What it doesn't do.** It has not shown a profit that can be told apart from zero. Read the verdict in section 5f for the current numbers.

**Why the hit rate is misleading.** The primary rule wins about two bets in three. That sounds like a winning model. It is break-even, because those bets are mostly favorites that need to win about two times in three just to pay for themselves.

**What limits it**

- **It tests against closing prices**, the sharpest ones. Lines posted earlier in the day are softer, and this data has no opening moneylines to test that.
- **It knows nothing about injuries, suspensions or travel.** The market does.
- **Older prices come from odds-listing sites**, with less vig than a real sportsbook charges.
- **Sportsbooks limit winning accounts**, and big moneyline favorites often have low limits or no price at all.

**Where an edge could still come from**

1. **Line shopping.** The same game is priced differently at different books. Taking the best of several prices is worth more than anything in this notebook, and it needs a live odds feed with several books.
2. **Earlier prices.** Log odds as they are posted and test whether the final probability beats the opening line.
3. **Information the ratings lack**, starting with who is actually playing.

**What not to do:** change the settings until the holdout looks better. It was looked at once, on purpose.